In [1]:
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

# 初始化环境
env = gym.make("ALE/SpaceInvaders-v5", render_mode="rgb_array")
env = DummyVecEnv([lambda: env])

# 定义回调函数（保存模型、定期评估）
checkpoint_callback = CheckpointCallback(
    save_freq=100_000,
    save_path="./logs1/",
    name_prefix="ppo_spaceinvaders"
)
eval_callback = EvalCallback(
    env,
    best_model_save_path="./logs1/",
    log_path="./logs1/",
    eval_freq=50_000,
    deterministic=True
)

model = PPO.load("ppo_spaceinvaders_500epochs", env=env, device="cuda")
# # 初始化模型（使用CnnPolicy处理图像输入）
# model = PPO(
#     "CnnPolicy",
#     env,
#     batch_size=128,
#     n_steps=512,
#     device="cuda",
#     verbose=1
# )

# 开始训练（总步数500万，200百万步）
model.learn(
    total_timesteps=2_000_000,
    reset_num_timesteps=False,
    callback=[checkpoint_callback, eval_callback]
)

# 保存最终模型
model.save("ppo_spaceinvaders_700epoch")

D:\anaconda\envs\pytorch\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: 'ppo_spaceinvaders_500epochs.zip'

In [ ]:
import gym
from stable_baselines3 import PPO
import time

# 加载训练好的模型
model = PPO.load("ppo_spaceinvaders_500epochs")
# model = PPO.load("logs/best_model")

# 创建环境并启用渲染（使用human模式显示画面）
env = gym.make("ALE/SpaceInvaders-v5", render_mode="human")

# 测试10个episode
for episode in range(10):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # 添加批次维度并将观测输入模型
        action, _ = model.predict(obs[None, ...], deterministic=True)
        
        # 执行动作
        obs, reward, terminated, truncated, info = env.step(action[0])
        done = terminated or truncated
        total_reward += reward
        
        # 添加微小延迟使画面可见（可选）
        time.sleep(0.01)
    
    print(f"Episode: {episode+1}, Total Reward: {total_reward}")

# 关闭环境
env.close()

D:\anaconda\envs\pytorch\lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [ ]:
import gym
from stable_baselines3 import PPO
import time
import numpy as np

# 加载训练好的模型
model = PPO.load("ppo_spaceinvaders_200epochs_256bs")

# 创建环境配置
env = gym.make(
    "ALE/SpaceInvaders-v5",
    render_mode="human",
)

# 初始化统计容器
episode_rewards = []
episode_durations = []  # 存活时间（步数）

# 测试10个episode
for episode in range(10):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    steps = 0
    
    while not done:
        # 添加批次维度并将观测输入模型
        action, _ = model.predict(obs[None, ...], deterministic=True)
        
        # 执行动作
        obs, reward, terminated, truncated, info = env.step(action[0])
        done = terminated or truncated
        
        total_reward += reward
        steps += 1
        
        # 添加微小延迟使画面可见
        time.sleep(0.01)
    
    # 计算指标
    survival_time = steps / 60  # 转换为秒（假设60FPS）
    
    # 记录数据
    episode_rewards.append(total_reward)
    episode_durations.append(survival_time)

    
    # 打印单局结果
    print(f"Episode {episode+1}:")
    print(f"  Total Reward: {total_reward}")
    print(f"  Survival Time: {survival_time:.2f}s")


# 计算综合统计数据
avg_reward = np.mean(episode_rewards)
std_reward = np.std(episode_rewards)
avg_duration = np.mean(episode_durations)


# 输出最终报告
print("\n=== Final Test Report ===")
print(f"Average Reward: {avg_reward:.1f} ± {std_reward:.1f}")
print(f"Average Survival Time: {avg_duration:.1f}s")


# 关闭环境
env.close()

D:\anaconda\envs\pytorch\lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: code() takes at most 16 arguments (18 given)
  warnings.warn(
D:\anaconda\envs\pytorch\lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() takes at most 16 arguments (18 given)
  warnings.warn(
D:\anaconda\envs\pytorch\lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [ ]:
# 添加Atari专用包装器
from stable_baselines3.common.atari_wrappers import AtariWrapper

env = AtariWrapper(
    env,
    frame_skip=4,       # 跳帧加速
    screen_size=84,      # 降采样
    grayscale=True,      # 灰度化
)
from stable_baselines3.common.vec_env import VecFrameStack

env = DummyVecEnv([lambda: gym.make(...)])
env = VecFrameStack(env, n_stack=4)  # 堆叠4帧